# Stochastic Processes Demo

Brownian motion, geometric Brownian motion, Poisson processes, and Markov chains.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.pardir, "src"))

import numpy as np
import matplotlib.pyplot as plt

from nms.stochastic import (
    brownian_motion,
    geometric_brownian_motion,
    poisson_process,
    compound_poisson_process,
    MarkovChain,
)

## 1. Brownian Motion

In [ ]:
T, n_steps, n_paths = 2.0, 1000, 8
t, W = brownian_motion(T, n_steps, n_paths=n_paths, seed=42)

plt.figure(figsize=(10, 5))
for i in range(n_paths):
    plt.plot(t, W[i], alpha=0.7, lw=0.7)
plt.fill_between(t, -2 * np.sqrt(t), 2 * np.sqrt(t), alpha=0.1, color="gray",
                 label=r"$\pm 2\sqrt{t}$ (95% envelope)")
plt.xlabel("t")
plt.ylabel("W(t)")
plt.title("Standard Brownian Motion")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Geometric Brownian Motion (Stock Price Model)

In [ ]:
S0, mu, sigma, T = 100, 0.08, 0.25, 2.0
t, S = geometric_brownian_motion(S0, mu, sigma, T, n_steps=500, n_paths=12, seed=7)

plt.figure(figsize=(10, 5))
for i in range(S.shape[0]):
    plt.plot(t, S[i], alpha=0.7, lw=0.7)
plt.plot(t, S0 * np.exp(mu * t), "k--", lw=2, label=r"$S_0 e^{\mu t}$ (expected path)")
plt.xlabel("t (years)")
plt.ylabel("S(t)")
plt.title(f"Geometric Brownian Motion (S₀={S0}, μ={mu}, σ={sigma})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Poisson Process

In [ ]:
rate, T = 5.0, 10.0
arrivals = poisson_process(rate, T, n_paths=5, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, times in enumerate(arrivals):
    counts = np.arange(1, len(times) + 1)
    axes[0].step(times, counts, where="post", alpha=0.8, label=f"Path {i+1}")
axes[0].set_xlabel("t")
axes[0].set_ylabel("N(t)")
axes[0].set_title(f"Poisson Process (λ = {rate})")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Compound Poisson
cpaths = compound_poisson_process(3.0, 10.0, jump_dist="normal",
                                   jump_params={"loc": 0, "scale": 2}, n_paths=5, seed=42)
for i, (times, cum_jumps) in enumerate(cpaths):
    if len(times) > 0:
        axes[1].step(times, cum_jumps, where="post", alpha=0.8)
axes[1].set_xlabel("t")
axes[1].set_ylabel("X(t)")
axes[1].set_title("Compound Poisson Process (normal jumps)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Markov Chain

In [ ]:
# Weather model: Sunny, Cloudy, Rainy
P = [
    [0.7, 0.2, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5],
]
mc = MarkovChain(P, state_labels=["Sunny", "Cloudy", "Rainy"])

states = mc.simulate(200, start=0, seed=42)
pi = mc.stationary_distribution()

print("Stationary distribution:")
for label, prob in zip(mc.state_labels, pi):
    print(f"  {label}: {prob:.4f}")

# Empirical frequencies from simulation
empirical = np.array([np.mean(states == i) for i in range(3)])
print("\nEmpirical frequencies (200 steps):")
for label, freq in zip(mc.state_labels, empirical):
    print(f"  {label}: {freq:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {0: "#f59e0b", 1: "#6b7280", 2: "#3b82f6"}
for i in range(len(states) - 1):
    axes[0].bar(i, 1, color=colors[states[i]], width=1.0, edgecolor="none")
axes[0].set_xlabel("Step")
axes[0].set_yticks([])
axes[0].set_title("Markov Chain Trajectory (Sunny/Cloudy/Rainy)")

x = np.arange(3)
axes[1].bar(x - 0.15, pi, 0.3, label="Stationary", color="#3b82f6", alpha=0.8)
axes[1].bar(x + 0.15, empirical, 0.3, label="Empirical", color="#f59e0b", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(mc.state_labels)
axes[1].set_ylabel("Probability")
axes[1].set_title("Stationary vs Empirical Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 5. Expected Hitting Times

In [ ]:
for target in range(3):
    h = mc.expected_hitting_time(target)
    print(f"Expected steps to reach {mc.state_labels[target]}:")
    for i, label in enumerate(mc.state_labels):
        print(f"  from {label}: {h[i]:.2f}")
    print()